In [1]:
import os
import json
import time
from openai import OpenAI
from litellm import token_counter

#open router key
os.environ["OPENROUTER_API_KEY"] = ""

# --- Client (OpenAI-compatible) ---
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

MODEL_NAME = "anthropic/claude-opus-4.6"

# Input/output file paths
INPUT_JSON = "toulmin_abstracts.json"
OUTPUT_JSON = "toulmin_consolidated.json"

# The fixed target claim (supplied externally)
TARGET_CLAIM = "Semaglutide induces significant weight loss in adults with obesity"


In [5]:
SYSTEM_PROMPT = """
You are a biomedical argumentation analyst. You synthesise a single 
consolidated Toulmin argument from multiple PubMed abstracts about a drug efficacy and 
safety claim.

Target Claim (Fixed, Ex Ante)
The following target claim is supplied externally:
Target claim: {TARGET_CLAIM}
You must not rewrite, reformulate, or replace this claim.

You will receive multiple abstracts at once, each identified by its PMID. Your task is to 
read all of them and produce ONE unified Toulmin argument where different abstracts 
contribute to different Toulmin elements. Every statement you make must cite the PMID(s) 
it draws from.

Toulmin Elements to Extract

1. Data
   Key empirical findings from across the abstracts that bear on the target claim.
   * Use short factual statements with effect magnitude where stated.
   * Each data point MUST include the PMID(s) it comes from.

2. Warrant
   The inferential link explaining why the data support the target claim.
   * If multiple abstracts share the same reasoning, consolidate into one warrant.
   * Cite the PMID(s) that express or imply this warrant.
   * Set type to "explicit" if stated in any abstract, "implicit" if you reconstructed it.

3. Backing
   Additional evidence that strengthens the warrant (e.g., systematic reviews or 
   meta-analyses that confirm the pattern seen in individual trials).
   * Cite PMID(s).

4. Qualifiers
   Statements from any abstract that limit the scope, population, duration, certainty, 
   or generalisability of the claim.
   * Each qualifier MUST include the PMID(s) it comes from.

5. Rebuttals
   Evidence from any abstract that challenges, weakens, or provides exceptions to the 
   target claim.
   * Each rebuttal MUST include the PMID(s) it comes from.

Output Format (Strict JSON Only)
Return one valid, parseable JSON object. No markdown fences, no preamble, no explanation.
Use this exact structure:

{
  "target_claim": "{TARGET_CLAIM}",
  "data": [
    {
      "finding": "<factual statement>",
      "pmids": ["<PMID1>", "<PMID2>"]
    }
  ],
  "warrant": {
    "type": "explicit | implicit",
    "content": "<inferential link>",
    "pmids": ["<PMID>"]
  },
  "backing": [
    {
      "evidence": "<supporting evidence>",
      "pmids": ["<PMID>"]
    }
  ],
  "qualifiers": [
    {
      "qualifier": "<limitation or scope restriction>",
      "pmids": ["<PMID>"]
    }
  ],
  "rebuttals": [
    {
      "rebuttal": "<counter-evidence or exception>",
      "pmids": ["<PMID>"]
    }
  ]
}

Constraints
* Do not introduce information not present in the abstracts.
* Every claim must cite at least one PMID.
* Do not assess clinical correctness or make policy recommendations.
* If an element has no evidence from any abstract, return an empty list for that element.
* Focus on mapping the structure of scientific argumentation, not on evaluating truth.
"""

In [3]:
def estimate_tokens(messages, model_id=MODEL_NAME):
    """
    Estimate prompt token count using LiteLLM's model-aware tokenizer.
    This lets us check we don't exceed the context window before sending.
    """
    return token_counter(model=model_id, messages=messages)

In [6]:
def call_llm(system_prompt, user_prompt, model_id=MODEL_NAME):
    """
    Send a system + user message to the LLM via OpenRouter.
    Returns the parsed JSON response and token usage stats.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    # Estimate tokens before sending
    estimated_tokens = estimate_tokens(messages, model_id)
    print(f"[LiteLLM estimate] prompt tokens ≈ {estimated_tokens}")

    # Safety check: warn if approaching context limits
    if estimated_tokens > 100000:
        print("[WARNING] Prompt is very large — consider reducing abstracts or using a model with larger context window.")

    # Send to OpenRouter
    response = client.chat.completions.create(
        model=model_id,
        messages=messages,
        extra_body={"reasoning": {"enabled": True}}     
    )

    # Extract response text
    raw_text = response.choices[0].message.content.strip()

    # Server-reported token usage
    usage = response.usage
    print(f"[Server usage] prompt={usage.prompt_tokens}, "
          f"completion={usage.completion_tokens}, "
          f"total={usage.total_tokens}")
    print(f"[Token diff] LiteLLM estimate vs server: "
          f"{estimated_tokens - usage.prompt_tokens}")

    # Parse JSON from response (strip markdown fences if present)
    cleaned = raw_text.replace("```json", "").replace("```", "").strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"[WARNING] JSON parse failed: {e}")
        print(f"[Raw response] {raw_text[:500]}")
        parsed = {"error": "JSON parse failed", "raw": raw_text}

    return parsed, {
        "estimated_prompt_tokens": estimated_tokens,
        "server_prompt_tokens": usage.prompt_tokens,
        "server_completion_tokens": usage.completion_tokens,
        "server_total_tokens": usage.total_tokens,
    }


# ============================================================
# 5. BUILD USER PROMPT FROM ALL ABSTRACTS
# ============================================================

def build_user_prompt(articles):
    """
    Construct one user message containing all abstracts.
    Each abstract is clearly delimited with its PMID and title.
    Only sends PMID, title, and abstract (not publication_types or toulmin_elements).
    """
    parts = []
    parts.append(f"Below are {len(articles)} PubMed abstracts. "
                 f"Synthesise one consolidated Toulmin argument from all of them.\n")

    for i, article in enumerate(articles, 1):
        parts.append(f"--- Abstract {i} ---")
        parts.append(f"PMID: {article['pmid']}")
        parts.append(f"Title: {article['title']}")
        parts.append(f"Abstract: {article['abstract']}")
        parts.append("")  # blank line separator

    return "\n".join(parts)


# ============================================================
# 6. MAIN PIPELINE
# ============================================================

def run_pipeline(input_path=INPUT_JSON, output_path=OUTPUT_JSON):
    """
    1. Load all abstracts from JSON
    2. Send all abstracts in one prompt to the LLM
    3. Receive one consolidated Toulmin argument
    4. Save result to JSON
    """
    # Load the abstracts
    with open(input_path, "r", encoding="utf-8") as f:
        articles = json.load(f)

    print(f"Loaded {len(articles)} articles from {input_path}")
    print(f"Model: {MODEL_NAME}")
    print(f"Target claim: {TARGET_CLAIM}\n")

    # Build the combined user prompt
    user_prompt = build_user_prompt(articles)

    # Call the LLM once with all abstracts
    print("Sending all abstracts to LLM for consolidated Toulmin extraction...\n")
    parsed, usage = call_llm(SYSTEM_PROMPT, user_prompt)

    # Wrap output with metadata
    output = {
        "metadata": {
            "model": MODEL_NAME,
            "num_abstracts": len(articles),
            "pmids_included": [a["pmid"] for a in articles],
            "target_claim": TARGET_CLAIM,
            "token_usage": usage,
        },
        "toulmin_argument": parsed,
    }

    # Save result
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"\nOutput saved to: {output_path}")


if __name__ == "__main__":
    run_pipeline()

Loaded 21 articles from toulmin_abstracts.json
Model: anthropic/claude-opus-4.6
Target claim: Semaglutide induces significant weight loss in adults with obesity

Sending all abstracts to LLM for consolidated Toulmin extraction...

[LiteLLM estimate] prompt tokens ≈ 11849
[Server usage] prompt=13444, completion=4579, total=18023
[Token diff] LiteLLM estimate vs server: -1595

Output saved to: toulmin_consolidated.json
